# Drug Review Insights & Summarization Tool: Main Pipeline

**Phase 5 — Integration & Testing**

**Dataset:** UCI ML Drug Review Dataset (Drugs.com) — 219,206 patient reviews covering 4,211 unique drugs across 2,724 conditions, with 10-star ratings, free-text reviews, and helpfulness votes spanning 2008–2017.

**Kaggle:** https://www.kaggle.com/datasets/jessicali9530/kuc-hackathon-winter-2018

**UCI Repository:** https://archive.ics.uci.edu/dataset/461/drug+review+dataset+druglib+com

**GitHub:** https://github.com/mvillanueva00/ADS-509-Final-Project

## Main Pipeline

This notebook is the master orchestration script for the Drug Review Insights & Summarization Tool. It runs all four pipeline notebooks in sequence from raw data ingestion through AI summarization, EDA, and final report generation.

Notebook: `05_main_pipeline.ipynb`

This notebook:
- Installs all required dependencies
- Sets up a standardized directory structure under `/content/data/`
- Runs Notebook 01: Data Ingestion & Preprocessing
- Runs Notebook 02: API Integration & Prompt Engineering
- Runs Notebook 03: EDA, Visualization & Business Analytics
- Runs Notebook 04: PDF Report Generation & AI Evaluation
- Validates all expected output files were produced
- Prints a final pipeline summary report

**Input files required:**
- `drugsComTrain_raw.csv` — Kaggle training split (161,297 rows)
- `drugsComTest_raw.csv` — Kaggle test split (53,766 rows)
- Gemini API key stored as a Colab secret named `GEMINI_API_KEY`
- All four pipeline notebooks uploaded to `/content/`

**Output files produced:**
- `cleaned_drug_reviews.csv` — fully cleaned and enriched dataset
- `drug_review_profile.csv` — dataset summary statistics
- `ai_summaries.json` — AI-generated narrative summaries
- `eda_summary_stats.csv` — EDA summary statistics
- `fig1` through `fig15` — EDA chart images
- `fig16_ai_evaluation.png` — AI qualitative evaluation bar chart
- `fig17_ai_evaluation_table.png` — AI qualitative evaluation table
- `drug_review_insights_report.pdf` — final combined report
- `evaluation_summary.csv` — AI accuracy evaluation table

## Step 1 — Mount Google Drive & Install Dependencies

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
# Install all required packages for the full pipeline
!pip install papermill kagglehub ucimlrepo reportlab wordcloud --quiet
print('All dependencies installed.')

All dependencies installed.


## Step 2 — Configure Pipeline Paths

All notebooks and input files must be uploaded to `/content/` before running. Update the paths below if your files are in a different location.

In [28]:
import os
import json
import time
import papermill as pm

# ── Pipeline configuration ─────────────────────────────────────────────────
# Set USE_CACHED_SUMMARIES = True to skip the Gemini API call
# and use the pre-generated ai_summaries.json instead.
# Set to False to run the full live pipeline with your Gemini API key.
USE_CACHED_SUMMARIES = False

# Root directory where all notebooks and files live
CONTENT_DIR = '/content'

# Input raw CSV files
TRAIN_CSV = os.path.join(CONTENT_DIR, 'drugsComTrain_raw.csv')
TEST_CSV  = os.path.join(CONTENT_DIR, 'drugsComTest_raw.csv')

# Create required directory structure for pipeline outputs
for d in ['/content/data/raw', '/content/data/processed', '/content/data/final']:
    os.makedirs(d, exist_ok=True)

# Patch Notebook 01 to use absolute paths instead of relative ../data/ paths
with open(os.path.join(CONTENT_DIR, '01_drug_review_preprocessing.ipynb'), 'r') as f:
    nb_str = f.read()

nb_str = nb_str.replace('../data/raw', '/content/data/raw')
nb_str = nb_str.replace('../data/processed', '/content/data/processed')
nb_str = nb_str.replace('../data/final', '/content/data/final')

with open(os.path.join(CONTENT_DIR, '01_drug_review_preprocessing_fixed.ipynb'), 'w') as f:
    f.write(nb_str)

# Notebook paths
NB_01 = os.path.join(CONTENT_DIR, '01_drug_review_preprocessing_fixed.ipynb')
NB_02 = os.path.join(CONTENT_DIR, '02_api_integration_prompt_engineering_fixed.ipynb')
NB_03 = os.path.join(CONTENT_DIR, '03_drug_review_eda.ipynb')
NB_04 = os.path.join(CONTENT_DIR, '04_pdf_and_evaluation.ipynb')

# Output notebook paths (papermill saves executed copies here)
OUT_DIR = os.path.join(CONTENT_DIR, 'pipeline_outputs')
os.makedirs(OUT_DIR, exist_ok=True)

OUT_NB_01 = os.path.join(OUT_DIR, '01_executed.ipynb')
OUT_NB_02 = os.path.join(OUT_DIR, '02_executed.ipynb')
OUT_NB_03 = os.path.join(OUT_DIR, '03_executed.ipynb')
OUT_NB_04 = os.path.join(OUT_DIR, '04_executed.ipynb')

print('Pipeline configuration ready.')
print(f'  USE_CACHED_SUMMARIES: {USE_CACHED_SUMMARIES}')
print(f'  Content directory: {CONTENT_DIR}')
print(f'  Output directory: {OUT_DIR}')
print('  Directory structure created.')
print('  Notebook 01 patched for absolute paths.')

Pipeline configuration ready.
  USE_CACHED_SUMMARIES: False
  Content directory: /content
  Output directory: /content/pipeline_outputs
  Directory structure created.
  Notebook 01 patched for absolute paths.


## Step 3 — Validate Input Files

Checks that all required input files and notebooks are present before running the pipeline.

In [29]:
import pandas as pd

print('Validating input files...\n')

required_files = {
    'Train CSV':        TRAIN_CSV,
    'Test CSV':         TEST_CSV,
    'Notebook 01':      NB_01,
    'Notebook 02':      NB_02,
    'Notebook 03':      NB_03,
    'Notebook 04':      NB_04,
}

all_present = True
for label, path in required_files.items():
    exists = os.path.exists(path)
    status = '✅' if exists else '❌ MISSING'
    print(f'  {status}  {label}: {os.path.basename(path)}')
    if not exists:
        all_present = False

print()
if all_present:
    # Quick row count check on raw CSVs
    train_rows = len(pd.read_csv(TRAIN_CSV))
    test_rows  = len(pd.read_csv(TEST_CSV))
    print(f'  Train CSV rows : {train_rows:,}')
    print(f'  Test CSV rows  : {test_rows:,}')
    print(f'  Combined total : {train_rows + test_rows:,}')
    print()
    print('All input files validated. Ready to run pipeline.')
else:
    print('Pipeline cannot start. Upload missing files to /content/ and re-run this cell.')

Validating input files...

  ✅  Train CSV: drugsComTrain_raw.csv
  ✅  Test CSV: drugsComTest_raw.csv
  ✅  Notebook 01: 01_drug_review_preprocessing_fixed.ipynb
  ✅  Notebook 02: 02_api_integration_prompt_engineering_fixed.ipynb
  ✅  Notebook 03: 03_drug_review_eda.ipynb
  ✅  Notebook 04: 04_pdf_and_evaluation.ipynb

  Train CSV rows : 161,297
  Test CSV rows  : 53,766
  Combined total : 215,063

All input files validated. Ready to run pipeline.


In [30]:
# Create required directory structure for Notebook 01
import os

dirs = [
    '/content/data/raw',
    '/content/data/processed',
    '/content/data/final',
]

for d in dirs:
    os.makedirs(d, exist_ok=True)
    print(f'Created: {d}')

print('Directory structure ready.')

Created: /content/data/raw
Created: /content/data/processed
Created: /content/data/final
Directory structure ready.


In [31]:
import json

# Patch Notebook 01 — replace relative paths with absolute /content/data/ paths
with open('/content/01_drug_review_preprocessing.ipynb', 'r') as f:
    nb = json.load(f)

nb_str = json.dumps(nb)
nb_str = nb_str.replace('../data/raw', '/content/data/raw')
nb_str = nb_str.replace('../data/processed', '/content/data/processed')
nb_str = nb_str.replace('../data/final', '/content/data/final')
nb_fixed = json.loads(nb_str)

with open('/content/01_drug_review_preprocessing_fixed.ipynb', 'w') as f:
    json.dump(nb_fixed, f)

print('Notebook 01 patched successfully.')
print('All relative paths replaced with absolute /content/data/ paths.')

# Patch Notebook 02 — patch cells directly by index
with open('/content/02_api_integration_prompt_engineering.ipynb', 'r') as f:
    nb02 = json.load(f)

# Patch Cell 3 — replace userdata call with temp file key read
nb02['cells'][3]['source'] = [
    'import google.generativeai as genai\n',
    '\n',
    'with open("/content/.gemini_key") as _f:\n',
    '    api_key = _f.read().strip()\n',
    'genai.configure(api_key=api_key)\n',
    'print("Gemini client ready.")'
]

# Patch Cell 15 — replace userdata call with temp file key read
nb02['cells'][15]['source'] = [
    'from google import genai\n',
    '\n',
    'with open("/content/.gemini_key") as _f:\n',
    '    api_key = _f.read().strip()\n',
    'print("Key found:", api_key is not None)\n',
    'print("Key preview:", api_key[:8] if api_key else "MISSING")\n',
    '\n',
    'client = genai.Client(api_key=api_key)\n',
    'print("Client:", client)'
]

with open('/content/02_api_integration_prompt_engineering_fixed.ipynb', 'w') as f:
    json.dump(nb02, f)

print('Notebook 02 patched for temp file API key injection.')

Notebook 01 patched successfully.
All relative paths replaced with absolute /content/data/ paths.
Notebook 02 patched for temp file API key injection.


## Step 4 — Run Notebook 01: Data Ingestion & Preprocessing

Loads Kaggle train/test CSVs, fetches UCI Druglib dataset via API, merges and cleans all data, engineers features, and exports the final cleaned dataset.

In [32]:
print('Running Notebook 01: Data Ingestion & Preprocessing...')
print('This may take a few minutes due to UCI Druglib API fetch and data cleaning.\n')

start = time.time()

try:
    pm.execute_notebook(
        NB_01,
        OUT_NB_01,
        kernel_name='python3'
    )
    elapsed = round(time.time() - start, 1)
    print(f'\n✅ Notebook 01 completed in {elapsed}s')

    # Copy outputs to /content/ for downstream notebooks
    import shutil
    for fname in ['cleaned_drug_reviews.csv', 'drug_review_profile.csv']:
        src = f'/content/data/final/{fname}'
        dst = f'/content/{fname}'
        if os.path.exists(src) and not os.path.exists(dst):
            shutil.copy(src, dst)
            print(f'  Copied {fname} to /content/')

except Exception as e:
    print(f'\n❌ Notebook 01 failed: {e}')
    print('Fix the error above before continuing.')

Running Notebook 01: Data Ingestion & Preprocessing...
This may take a few minutes due to UCI Druglib API fetch and data cleaning.



Executing:   0%|          | 0/67 [00:00<?, ?cell/s]


✅ Notebook 01 completed in 79.7s


## Step 5 — Run Notebook 02: API Integration & Prompt Engineering

Builds structured data profiles, sends them to the Gemini API, and saves AI-generated summaries to `ai_summaries.json`.

Set `USE_CACHED_SUMMARIES = True` in Step 2 to skip the API call and use pre-generated summaries.

In [33]:
if USE_CACHED_SUMMARIES:
    cached_path = os.path.join(CONTENT_DIR, 'ai_summaries.json')
    if os.path.exists(cached_path):
        print('USE_CACHED_SUMMARIES = True')
        print('✅ Using pre-generated ai_summaries.json — skipping Gemini API call.')
    else:
        print('❌ USE_CACHED_SUMMARIES = True but ai_summaries.json not found in /content/')
        print('Upload ai_summaries.json to /content/ or set USE_CACHED_SUMMARIES = False.')
else:
    print('Running Notebook 02: API Integration & Prompt Engineering...')
    print('This will make live Gemini API calls.\n')

    # Retrieve key from Colab secrets and write to a temp file
    # that the patched Notebook 02 will read
    from google.colab import userdata
    gemini_key = userdata.get('GEMINI_API_KEY')

    key_file = '/content/.gemini_key'
    with open(key_file, 'w') as f:
        f.write(gemini_key)

    start = time.time()

    try:
        pm.execute_notebook(
            NB_02,
            OUT_NB_02,
            kernel_name='python3'
        )
        elapsed = round(time.time() - start, 1)
        print(f'\n✅ Notebook 02 completed in {elapsed}s')

        # Clean up key file
        if os.path.exists(key_file):
            os.remove(key_file)

        import shutil
        src = '/content/data/final/ai_summaries.json'
        dst = '/content/ai_summaries.json'
        if os.path.exists(src) and not os.path.exists(dst):
            shutil.copy(src, dst)
            print(f'  Copied ai_summaries.json to /content/')

    except Exception as e:
        print(f'\n❌ Notebook 02 failed: {e}')
        print('Check that your GEMINI_API_KEY Colab secret is set and valid.')

Running Notebook 02: API Integration & Prompt Engineering...
This will make live Gemini API calls.



Executing:   0%|          | 0/26 [00:00<?, ?cell/s]


✅ Notebook 02 completed in 292.0s
  Copied ai_summaries.json to /content/


## Step 6 — Run Notebook 03: EDA, Visualization & Business Analytics

Performs exploratory data analysis, generates all 15 chart images, and exports `eda_summary_stats.csv`.

In [34]:
print('Running Notebook 03: EDA, Visualization & Business Analytics...')
print('This generates all 15 chart images. May take a few minutes.\n')

start = time.time()

try:
    pm.execute_notebook(
        NB_03,
        OUT_NB_03,
        kernel_name='python3'
    )
    elapsed = round(time.time() - start, 1)
    print(f'\n✅ Notebook 03 completed in {elapsed}s')

except Exception as e:
    print(f'\n❌ Notebook 03 failed: {e}')
    print('Fix the error above before continuing.')

Running Notebook 03: EDA, Visualization & Business Analytics...
This generates all 15 chart images. May take a few minutes.



Executing:   0%|          | 0/50 [00:00<?, ?cell/s]

ERROR:papermill:unhandled iopub msg: colab_request
ERROR:papermill:unhandled iopub msg: colab_request
ERROR:papermill:unhandled iopub msg: colab_request



✅ Notebook 03 completed in 172.3s


## Step 7 — Run Notebook 04: PDF Report Generation & AI Evaluation

Combines AI narratives with EDA charts into a final PDF report. Produces AI evaluation figures and written reflection.

In [35]:
print('Running Notebook 04: PDF Report Generation & AI Evaluation...')
print('This combines all outputs into the final deliverable report.\n')

start = time.time()

try:
    pm.execute_notebook(
        NB_04,
        OUT_NB_04,
        kernel_name='python3'
    )
    elapsed = round(time.time() - start, 1)
    print(f'\n✅ Notebook 04 completed in {elapsed}s')

except Exception as e:
    print(f'\n❌ Notebook 04 failed: {e}')
    print('Fix the error above before continuing.')

Running Notebook 04: PDF Report Generation & AI Evaluation...
This combines all outputs into the final deliverable report.



Executing:   0%|          | 0/19 [00:00<?, ?cell/s]

ERROR:papermill:unhandled iopub msg: colab_request
ERROR:papermill:unhandled iopub msg: colab_request
ERROR:papermill:unhandled iopub msg: colab_request



✅ Notebook 04 completed in 137.8s


## Step 8 — Validate Pipeline Outputs

Checks that all expected output files were produced successfully.

In [36]:
print('Validating pipeline outputs...\n')

expected_outputs = {
    'cleaned_drug_reviews.csv':         '/content/cleaned_drug_reviews.csv',
    'drug_review_profile.csv':          '/content/drug_review_profile.csv',
    'ai_summaries.json':                '/content/ai_summaries.json',
    'eda_summary_stats.csv':            '/content/eda_summary_stats.csv',
    'drug_review_insights_report.pdf':  '/content/drug_review_insights_report.pdf',
    'evaluation_summary.csv':           '/content/evaluation_summary.csv',
    'fig16_ai_evaluation.png':          '/content/fig16_ai_evaluation.png',
    'fig17_ai_evaluation_table.png':    '/content/fig17_ai_evaluation_table.png',
}

# Check EDA chart images fig1 through fig15
for i in range(1, 16):
    key = f'fig{i}_*.png'
    # Check if any file matching fig{i}_*.png exists
    matches = [f for f in os.listdir('/content') if f.startswith(f'fig{i}_') and f.endswith('.png')]
    expected_outputs[f'fig{i} chart'] = matches[0] if matches else None

all_good = True
for label, path in expected_outputs.items():
    if path and os.path.exists(path):
        size_kb = round(os.path.getsize(path) / 1024, 1)
        print(f'  ✅  {label} ({size_kb} KB)')
    else:
        print(f'  ❌  {label} — NOT FOUND')
        all_good = False

print()
if all_good:
    print('✅ All pipeline outputs validated successfully.')
else:
    print('❌ Some outputs are missing. Check the failed notebook steps above.')

Validating pipeline outputs...

  ✅  cleaned_drug_reviews.csv (121462.4 KB)
  ✅  drug_review_profile.csv (0.2 KB)
  ✅  ai_summaries.json (40.1 KB)
  ✅  eda_summary_stats.csv (0.4 KB)
  ✅  drug_review_insights_report.pdf (1260.9 KB)
  ✅  evaluation_summary.csv (0.3 KB)
  ✅  fig16_ai_evaluation.png (73.4 KB)
  ✅  fig17_ai_evaluation_table.png (459.7 KB)
  ✅  fig1 chart (78.1 KB)
  ✅  fig2 chart (125.0 KB)
  ✅  fig3 chart (75.7 KB)
  ✅  fig4 chart (55.8 KB)
  ✅  fig5 chart (68.7 KB)
  ✅  fig6 chart (45.3 KB)
  ✅  fig7 chart (84.3 KB)
  ✅  fig8 chart (44.1 KB)
  ✅  fig9 chart (149.2 KB)
  ✅  fig10 chart (134.4 KB)
  ✅  fig11 chart (88.0 KB)
  ✅  fig12 chart (66.8 KB)
  ✅  fig13 chart (104.5 KB)
  ✅  fig14 chart (705.7 KB)
  ✅  fig15 chart (223.5 KB)

✅ All pipeline outputs validated successfully.


## Step 9 — Pipeline Summary Report

In [37]:
import pandas as pd

print('=' * 60)
print('  DRUG REVIEW INSIGHTS & SUMMARIZATION TOOL')
print('  Pipeline Execution Summary')
print('=' * 60)
print()

# Dataset stats
try:
    df = pd.read_csv('/content/cleaned_drug_reviews.csv', low_memory=False)
    print(f'  Dataset rows        : {len(df):,}')
    print(f'  Unique drugs        : {df["drugName"].nunique():,}')
    print(f'  Unique conditions   : {df["condition"].nunique():,}')
    print(f'  Average rating      : {df["rating"].mean():.2f} / 10')
    print()
except Exception:
    print('  Could not load cleaned dataset for summary.')
    print()

# AI summaries stats
try:
    with open('/content/ai_summaries.json', 'r') as f:
        summaries = json.load(f)
    print(f'  AI drugs summarized     : {len(summaries["drugs"])}')
    print(f'  AI conditions summarized: {len(summaries["conditions"])}')
    print()
except Exception:
    print('  Could not load AI summaries for stats.')
    print()

# Output files
print('  Output files:')
output_files = [
    'cleaned_drug_reviews.csv',
    'drug_review_profile.csv',
    'ai_summaries.json',
    'eda_summary_stats.csv',
    'drug_review_insights_report.pdf',
    'evaluation_summary.csv',
    'fig16_ai_evaluation.png',
    'fig17_ai_evaluation_table.png',
]
for fname in output_files:
    path = f'/content/{fname}'
    if os.path.exists(path):
        size_kb = round(os.path.getsize(path) / 1024, 1)
        print(f'    ✅ {fname} ({size_kb} KB)')
    else:
        print(f'    ❌ {fname} — missing')

print()
print('=' * 60)
print('  Pipeline complete.')
print('=' * 60)

  DRUG REVIEW INSIGHTS & SUMMARIZATION TOOL
  Pipeline Execution Summary

  Dataset rows        : 219,206
  Unique drugs        : 4,211
  Unique conditions   : 2,644
  Average rating      : 6.99 / 10

  AI drugs summarized     : 5
  AI conditions summarized: 5

  Output files:
    ✅ cleaned_drug_reviews.csv (121462.4 KB)
    ✅ drug_review_profile.csv (0.2 KB)
    ✅ ai_summaries.json (40.1 KB)
    ✅ eda_summary_stats.csv (0.4 KB)
    ✅ drug_review_insights_report.pdf (1260.9 KB)
    ✅ evaluation_summary.csv (0.3 KB)
    ✅ fig16_ai_evaluation.png (73.4 KB)
    ✅ fig17_ai_evaluation_table.png (459.7 KB)

  Pipeline complete.
